In [1]:
import ee
import pandas as pd
import json
from glob import glob
import geemap
import pyproj
from geopandas import geopandas as gpd
from shapely.geometry import Point
from copy import deepcopy
import numpy as np
import os
import ast
from google.cloud import storage

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
ee.Authenticate()
ee_project = "corestack1-dev-alpha"
ee.Initialize(project="core-stack-dev-2")#"corestack1-dev-alpha")

In [4]:
client = storage.Client()
bucket = client.get_bucket('core_stack')

In [5]:
best_month_dict = {'Eastern Plateau & Hills Region': 'cc_12',
                   'Middle Gangetic Plain Region': 'cc_10',
                   'Lower Gangetic Plain Region': 'cc_9',
                   'Western Himalayan Region': 'cc_8',
                   'Eastern Himalayan Region': 'cc_10',
                   'Upper Gangetic Plain Region': 'cc_9',
                   'Trans Gangetic Plain Region': 'cc_9',
                   'Central Plateau & Hills Region': 'cc_7',
                   'Western Plateau and Hills Region': 'cc_11',
                   'Southern Plateau and Hills Region': 'cc_8',
                   'East Coast Plains & Hills Region': 'cc_12'}

In [6]:
agroclimaticZone_acronym_dict = {'Eastern Plateau & Hills Region': 'EPAHR',
                               'Southern Plateau and Hills Region': 'SPAHR',
                               'East Coast Plains & Hills Region': 'ECPHR',
                               'Western Plateau and Hills Region': 'WPAHR',
                               'Central Plateau & Hills Region': 'CPAHR',
                               'Lower Gangetic Plain Region': 'LGPR',
                                'Middle Gangetic Plain Region': 'MGPR',
                                'Eastern Himalayan Region': 'EHR',
                                'Western Himalayan Region': 'WHR',
                                'Upper Gangetic Plain Region': 'UGPR',
                                'Trans Gangetic Plain Region': 'TGPR',
                                'West Coast Plains & Ghat Region': 'WCPGR',
                                'Gujarat Plains & Hills Region': 'GPHR',
                                'Western Dry Region': 'WDR'}

In [7]:
def upload_file_to_gcs(local_file_path, file_name):
    blob = bucket.blob(f'nrm_tree_health/correction_compiled_results/{file_name}.csv')        # GCS path
    blob.upload_from_filename(local_file_path)

    print("Upload complete.")

def export_to_gee(file_name):
  # CSV GCS path
  gcs_path = f'gs://core_stack/nrm_tree_health/correction_compiled_results/{file_name}.csv'

  # Create task ID and run table ingestion
  task_id = ee.data.newTaskId()[0]
  asset_id = f'projects/{ee_project}/assets/tree_characteristics/{file_name}'

  manifest = {
    'id': asset_id,
    'sources': [
        {
            'primaryPath': gcs_path,
            'additionalPaths': []
        }
    ]
  }

  ee.data.startTableIngestion(task_id, manifest)
  print("Ingestion task started:", task_id)

Merge Correction CSVs

In [9]:
acz_list = [
    # 'Western Himalayan Region',
    'Eastern Himalayan Region',
    # 'Lower Gangetic Plain Region',
    # 'Middle Gangetic Plain Region',
    # 'Upper Gangetic Plain Region',
    # 'Trans Gangetic Plain Region',
    # 'Eastern Plateau & Hills Region',
    # 'Central Plateau & Hills Region',
    # 'Western Plateau and Hills Region',
    # 'Southern Plateau and Hills Region',
    # 'East Coast Plains & Hills Region'
    ]



In [10]:
years = ['2017', '2018', '2019', '2020', '2021', '2022', '2023']

# CCD

In [ ]:
# CCD

for agroclimatic_zone in acz_list:
    print(agroclimatic_zone)
    for year in years:
      print(year)
      df = pd.read_csv(f'drive/MyDrive/TreeHealth/Agroclimatic_regions/{agroclimatic_zone}.csv')
      dist_list = list(df['Name'])
      print(f'length(dist_list): {len(dist_list)}')

      i = 0
      merged_df = pd.DataFrame()
      for district in dist_list:
          print(i, district)
          i += 1

          try:
            df = pd.read_csv(f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{year}/result_monthly_cc_corrections.csv')
          except:
              continue

          merged_df = pd.concat([merged_df, df], ignore_index=True)

      print(f'length(merged_df): {len(merged_df)}')

      cols = merged_df.columns
      for i in range(1, len(cols)):
          merged_df[cols[i]] = merged_df[cols[i]].astype('Int64')
      drive_path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/corrections_{year}.csv'
      merged_df.to_csv(drive_path, index=False)

      file_name = f"corrections_ccd_{year}_result_0_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
      upload_file_to_gcs(drive_path, file_name)
      export_to_gee(file_name)

Eastern Plateau & Hills Region
2020
length(dist_list): 119
0 East Godavari
1 Srikakulam
2 Visakhapatnam
3 Vizianagaram
4 AurangabadB
5 Banka
6 Gaya
7 Jamui
8 Nawada
9 Rohtas
10 Baloda Bazar
11 Balod
12 BalrampurC
13 Bastar
14 Bemetara
15 BijapurC


# CH

In [11]:
# CH

# Function to convert string representation of list to an actual list
def convert_to_list(string):
    return ast.literal_eval(string)

for agroclimatic_zone in acz_list:
    print(agroclimatic_zone)
    for year in years:
      print(year)
      df = pd.read_csv('drive/MyDrive/TreeHealth/district_to_agroclimaticZone_mapping.csv')
      df['IntersectingZones'] = df['IntersectingZones'].apply(convert_to_list)
      district_mapping_df = df[df['AgroclimaticZone'] == agroclimatic_zone][['District', 'IntersectingZones']]
      dist_list =  []
      for ind in district_mapping_df.index:
          district = district_mapping_df.loc[ind, 'District']
          zones = district_mapping_df['IntersectingZones'][ind]
          dist_list.append(district)

      print(f'length(dist_list): {len(dist_list)}')

      correction_num = 0
      j = 0
      merged_df = pd.DataFrame()


      for district in dist_list:
          print(j, district)
          j += 1

          try:
            df = pd.read_csv(f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{year}/result_chm_corrections.csv')
          except:
              continue

          merged_df = pd.concat([merged_df, df], ignore_index=True)
          print(f'length(merged_df): {len(merged_df)}')
          drive_path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/corrections_chm_{year}_{correction_num}.csv'

          if len(merged_df) > 50000000: # Adjust based on need, if script crashes because of RAM exhaustion, default 70000000
              cols = merged_df.columns
              for i in range(1, len(cols)):
                  merged_df[cols[i]] = merged_df[cols[i]].astype('Int64')
              print(f'Saving merged_corrections_{correction_num}.csv')
              merged_df.to_csv(drive_path, index=False)

              file_name = f"corrections_ch_{year}_result_{correction_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
              upload_file_to_gcs(drive_path, file_name)
              export_to_gee(file_name)

              del(merged_df)
              correction_num += 1
              merged_df = pd.DataFrame()

      if len(merged_df) > 0:
          cols = merged_df.columns
          for i in range(1, len(cols)):
              merged_df[cols[i]] = merged_df[cols[i]].astype('Int64')
          print(f'Saving merged_corrections_{correction_num}.csv')
          merged_df.to_csv(drive_path, index=False)

          file_name = f"corrections_ch_{year}_result_{correction_num}_{agroclimaticZone_acronym_dict[agroclimatic_zone]}"
          upload_file_to_gcs(drive_path, file_name)
          export_to_gee(file_name)

          del(merged_df)

Eastern Himalayan Region
2017
length(dist_list): 99
0 Anjaw
length(merged_df): 2979532
1 Changlang
length(merged_df): 3307447
2 Dibang Valley
length(merged_df): 4194324
3 East Kameng
length(merged_df): 4352964
4 East Siang
length(merged_df): 4458119
5 Kurung Kumey
6 Lohit
length(merged_df): 5274731
7 Longding
length(merged_df): 5300151
8 Lower Dibang Valley
length(merged_df): 6630346
9 Lower Subansiri
10 Namsai
length(merged_df): 6654526
11 Papum Pare
12 Tawang
13 Tirap
length(merged_df): 6808687
14 Upper Siang
15 Upper Subansiri
16 West Kameng
length(merged_df): 7216607
17 West Siang
18 Baksa
length(merged_df): 7300480
19 Barpeta
length(merged_df): 7319087
20 Bongaigaon
length(merged_df): 7335411
21 Cachar
length(merged_df): 7372150
22 Chirang
length(merged_df): 7420786
23 Darrang
length(merged_df): 7454804
24 Dhemaji
length(merged_df): 7457716
25 Dhubri
length(merged_df): 7488291
26 Dibrugarh
length(merged_df): 7520020
27 Dima Hasao
length(merged_df): 7968962
28 Goalpara
length(merge

Upload complete.
Ingestion task started: 719ee0b0-b6f9-4f82-a567-e1de6a176ab3
65 Champhai
length(merged_df): 3634458
66 Kolasib
length(merged_df): 4210001
67 Lawangtlai
length(merged_df): 5163272
68 Lunglei
length(merged_df): 7603186
69 Mamit
length(merged_df): 8935328
70 Saiha
length(merged_df): 10161350
71 Serchhip
length(merged_df): 10989166
72 Dimapur
length(merged_df): 11231066
73 Kiphire
length(merged_df): 11796555
74 Kohima
length(merged_df): 12588621
75 Longleng
length(merged_df): 12837079
76 Mokokchung
length(merged_df): 13936156
77 Mon
length(merged_df): 14914721
78 Peren
length(merged_df): 15919623
79 Phek
length(merged_df): 17010418
80 Tuensang
length(merged_df): 18037322
81 Wokha
length(merged_df): 19164284
82 Zunheboto
length(merged_df): 20101345
83 East Sikkim
84 North Sikkim
85 South Sikkim
length(merged_df): 20114100
86 West Sikkim
length(merged_df): 20114504
87 Dhalai
length(merged_df): 20852781
88 Gomati
length(merged_df): 21287454
89 Khowai
length(merged_df): 217055

Upload complete.
Ingestion task started: 5a592b45-4bd1-4fef-9018-cd77a083c6de
2023
length(dist_list): 99
0 Anjaw
length(merged_df): 1848049
1 Changlang
length(merged_df): 4620455
2 Dibang Valley
length(merged_df): 7067896
3 East Kameng
length(merged_df): 9238528
4 East Siang
length(merged_df): 10273774
5 Kurung Kumey
length(merged_df): 12139221
6 Lohit
length(merged_df): 13411891
7 Longding
length(merged_df): 13957060
8 Lower Dibang Valley
length(merged_df): 15800235
9 Lower Subansiri
length(merged_df): 17537049
10 Namsai
length(merged_df): 17592906
11 Papum Pare
length(merged_df): 19454295
12 Tawang
length(merged_df): 20259605
13 Tirap
length(merged_df): 20849587
14 Upper Siang
length(merged_df): 25013318
15 Upper Subansiri
length(merged_df): 26200628
16 West Kameng
length(merged_df): 29142728
17 West Siang
length(merged_df): 31692865
18 Baksa
length(merged_df): 31844938
19 Barpeta
length(merged_df): 31948301
20 Bongaigaon
length(merged_df): 32020012
21 Cachar
length(merged_df): 33015